<!-- Semantic Search with Pincone & Embeddings using Hugingface Model -->

In [2]:
%pip install pinecone
%pip install sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 742.8/742.8 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.7/280.7 kB 33.0 MB/s eta 0:00:00


In [3]:
# import Libraries
import pandas as pd 
import os 
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from tqdm import tqdm
import pinecone

In [4]:
from google.colab import drive
drive.mount('/content/drive')
file_path = '/content/drive/MyDrive/pincone/articles_new.csv'
df = pd.read_csv(file_path)
df
df['class'] = ['class-a', 'class-b'] * 250
df


Mounted at /content/drive


,title,id,class
0,Mental Note Vol. 24,3054,class-a
1,Your Brain On Coronavirus,3055,class-b
2,Mind Your Nose,3056,class-a
3,The 4 Purposes of Dreams,3057,class-b
4,Surviving a Rod Through the Head,3058,class-a
...,...,...,...
495,Is It Worth to Invest In Mobile E-commerce App...,3549,class-b
496,Let go of these things for a happier 2021,3550,class-a
497,Not Everyone Will like Your Writing,3551,class-b
498,Is Technology Neutral?,3552,class-a


Embedding Using Hugingface Model

In [5]:
Model_huggingface = SentenceTransformer(model_name_or_path='BAAI/bge-small-en-v1.5', device='cpu')
Model_huggingface

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'cls', 'include_prompt': True})
  (2): Normalize({})
)

In [6]:
## Embedding for instance
Vect_Legnth = Model_huggingface.encode(df['title'].iloc[0])
print(f"Vector_Length: {len(Vect_Legnth)}")
## Frist 10 values in Embedding
Model_huggingface.encode(df['title'].iloc[0])[:10]

Vector_Length: 384


array([-0.03699718,  0.02828607,  0.00319821, -0.00102526,  0.01951823,
        0.02105058, -0.05418623,  0.02383843, -0.01735727,  0.0149247 ],
      dtype=float32)

In [ ]:
## Load dotenv file
# load_dotenv(override=True)
# Pincone_API_KEY = os.getenv('PINECONE_API_KEY')
# Pincone_Host = os.getenv('PINCONE_HOST')

In [ ]:
# conecting with pinecone
from pinecone import Pinecone

pc = Pinecone(api_key='YOUR_API_KEY')

index = pc.Index("semantic-search") 

print(index.describe_index_stats())


{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '185',
                                    'content-type': 'application/json',
                                    'date': 'Mon, 04 May 2026 09:32:57 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '73',
                                    'x-pinecone-request-latency-ms': '72',
                                    'x-pinecone-response-duration-ms': '74'}},
 'dimension': 384,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 490}},
 'storageFullness': 0.0,
 'total_vector_count': 490,
 'vector_type': 'dense'}


In [9]:
# Looping over the Dataset and upsert through batches
from turtle import title


batch_size = 32
failed_ids = []
for batch_start in tqdm(range(0, len(df), batch_size)):  
    try:
        batch_end = min(batch_start + batch_size, len(df))   # to handle the end of each batch
        titles_batch = df['title'][batch_start:batch_end].tolist() 
        ids_batch = df['id'][batch_start:batch_end].astype(str).tolist()
        metadata_batch = df['class'][batch_start:batch_end].tolist()
        ## Get Embeddings using HuggingFace model
        embeds_batches = Model_huggingface.encode(titles_batch)

        ## Upsert to Pincone
        to_upsert = [(id, emb, {'class': cls, 'title': title})
        for id, emb, cls, title in zip(ids_batch, embeds_batches, metadata_batch, titles_batch)]
        
        _=index.upsert(vectors=to_upsert)
        
    except Exception as e:
        print(f"Error processing batch starting at index{e}")
        failed_ids.append(ids_batch)  # Collect IDs of failed batches for later review

100%|██████████| 16/16 [00:14<00:00,  1.10it/s]


In [10]:
## Query
query_text = "Mental Note Vol. 24"

# Get embedding for the query
query_embedding = Model_huggingface.encode(query_text).tolist()

## search in pinecone
search_results = index.query(vector=query_embedding, top_k=5, include_metadata=True,    include_values=True)
print(search_results)

QueryResponse(matches=[{'id': '3054',
 'metadata': {'class': 'class-a', 'title': 'Mental Note Vol. 24'},
 'score': 1.00121307,
 'values': [-0.0369971395,
            0.0282860957,
            0.00319820899,
            -0.0010252503,
            0.0195182823,
            0.0210505798,
            -0.054186251,
            0.0238383729,
            -0.0173572432,
            0.0149247684,
            -0.0187694803,
            0.0711696818,
            0.0108696492,
            0.00914127845,
            -0.0113440342,
            0.022315681,
            -0.0686033219,
            -0.0170724802,
            -0.0477301516,
            0.0279178023,
            0.0177194104,
            -0.0231913514,
            0.0369938314,
            0.0335371234,
            -0.0176938176,
            -0.0575827546,
            -0.0190309957,
            -0.0352554768,
            -0.0299814399,
            -0.115479097,
            -0.0298884679,
            -0.0403272361,
            0.0268735979

In [15]:
## delete vectors from pincone
_=index.delete(ids =['3054', '3055', '3056', '3057', '3058', '3059', '3060', '3061', '3062', '3063'])